## Example 1: The plugin is registered with pytest

In [ ]:
import importlib.metadata

import pytest_randomly

# Example 1: the pytest11 entry point is what makes it switch itself on in every pytest run
assert importlib.metadata.version("pytest-randomly") == "5.0.0"
eps = {ep.name: ep.value for ep in importlib.metadata.entry_points(group="pytest11")}
assert eps.get("randomly") == "pytest_randomly", eps
assert callable(pytest_randomly.pytest_addoption) and callable(pytest_randomly.pytest_collection_modifyitems)
print("Example 1 OK: pytest11 randomly =", eps["randomly"])

## Example 2: Tests run shuffled, and a seed replays the order

In [ ]:
import pathlib
import subprocess
import sys
import tempfile

def run_tests(*args):
    """Run 12 tiny tests in a fresh folder; return the order they ran in (and the random value each saw)."""
    d = pathlib.Path(tempfile.mkdtemp())
    (d / "test_order.py").write_text("import random, pathlib\n" + "".join(
        f"def test_{i:02d}():\n    pathlib.Path(r'{d}/log').open('a').write('{i:02d} %r\\n' % random.random())\n" for i in range(12)))
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-o", "addopts=", "-p", "no:cacheprovider", *args, str(d)],
                       capture_output=True, text=True, cwd=d)
    assert r.returncode == 0, r.stdout + r.stderr
    rows = [line.split(" ", 1) for line in (d / "log").read_text().splitlines()]
    return [n for n, _ in rows], dict(rows), r.stdout

# Example 2: the same --randomly-seed gives the same shuffled order; another seed gives another
first, _, out = run_tests("-p", "randomly", "--randomly-seed=1234")
again, _, _ = run_tests("-p", "randomly", "--randomly-seed=1234")
other, _, _ = run_tests("-p", "randomly", "--randomly-seed=99")
assert first == again and first != sorted(first) and other != first, (first, other)
assert "12 passed" in out, out[-300:]
print("Example 2 OK:", first)

## Example 3: Python's random is reseeded per test, the same way every run

In [ ]:
import pathlib
import subprocess
import sys
import tempfile

def run_tests(*args):
    """Run 12 tiny tests in a fresh folder; return the order they ran in (and the random value each saw)."""
    d = pathlib.Path(tempfile.mkdtemp())
    (d / "test_order.py").write_text("import random, pathlib\n" + "".join(
        f"def test_{i:02d}():\n    pathlib.Path(r'{d}/log').open('a').write('{i:02d} %r\\n' % random.random())\n" for i in range(12)))
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-o", "addopts=", "-p", "no:cacheprovider", *args, str(d)],
                       capture_output=True, text=True, cwd=d)
    assert r.returncode == 0, r.stdout + r.stderr
    rows = [line.split(" ", 1) for line in (d / "log").read_text().splitlines()]
    return [n for n, _ in rows], dict(rows), r.stdout

# Example 3: before each test random is seeded from the run's seed + a hash of the test id, so a test sees the
# same numbers whatever order it runs in; --randomly-dont-reset-seed stops that
_, a, _ = run_tests("-p", "randomly", "--randomly-seed=7")
_, b, _ = run_tests("-p", "randomly", "--randomly-seed=7", "--randomly-dont-reorganize")
assert a == b and len(set(a.values())) == 12, (a, b)
_, c, _ = run_tests("-p", "randomly", "--randomly-seed=7", "--randomly-dont-reset-seed")
assert c != a, c
print("Example 3 OK: test_00 always sees", a["00"])

## Example 4: Switch it off, or keep the order

In [ ]:
import pathlib
import subprocess
import sys
import tempfile

def run_tests(*args):
    """Run 12 tiny tests in a fresh folder; return the order they ran in (and the random value each saw)."""
    d = pathlib.Path(tempfile.mkdtemp())
    (d / "test_order.py").write_text("import random, pathlib\n" + "".join(
        f"def test_{i:02d}():\n    pathlib.Path(r'{d}/log').open('a').write('{i:02d} %r\\n' % random.random())\n" for i in range(12)))
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-o", "addopts=", "-p", "no:cacheprovider", *args, str(d)],
                       capture_output=True, text=True, cwd=d)
    assert r.returncode == 0, r.stdout + r.stderr
    rows = [line.split(" ", 1) for line in (d / "log").read_text().splitlines()]
    return [n for n, _ in rows], dict(rows), r.stdout

# Example 4: -p no:randomly turns the plugin off; --randomly-dont-reorganize keeps the order but still reseeds
off, _, _ = run_tests("-p", "no:randomly")
assert off == sorted(off), off
kept, kept_values, _ = run_tests("-p", "randomly", "--randomly-dont-reorganize", "--randomly-seed=5")
_, shuffled_values, _ = run_tests("-p", "randomly", "--randomly-seed=5")
assert kept == sorted(kept) and kept_values == shuffled_values, kept
print("Example 4 OK")

## Example 5: NumPy's global random state is reseeded too

In [ ]:
import pathlib
import subprocess
import sys
import tempfile

# Example 5: with numpy installed, np.random's legacy global state is reseeded per test as well, so two runs agree
def run():
    d = pathlib.Path(tempfile.mkdtemp())
    (d / "test_np.py").write_text("import numpy as np, pathlib\n" + "".join(
        f"def test_{i}():\n    pathlib.Path(r'{d}/log').open('a').write('{i} %r\\n' % np.random.random())\n" for i in range(5)))
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-o", "addopts=", "-p", "no:cacheprovider", "-p", "randomly",
                        "--randomly-seed=3", str(d)], capture_output=True, text=True, cwd=d)
    assert r.returncode == 0, r.stdout + r.stderr
    return dict(line.split(" ", 1) for line in (d / "log").read_text().splitlines())
first, second = run(), run()
assert first == second and len(set(first.values())) == 5, (first, second)
print("Example 5 OK:", first)